# 06 · Boyut İndirgeme
Çok sayıda özelliği daha az sayıda, bilgiyi koruyan bileşene indirgemek.

| Yöntem | Tür | Kullanım |
|---|---|---|
| **PCA** | Doğrusal | Özellik sayısını azaltma, gürültü giderme, çoklu doğrusallığı çözme, model girdisi |
| **t-SNE / UMAP** | Doğrusal olmayan | Sadece **görselleştirme** (yeni veriye uygulanamaz, mesafeler yorumlanamaz) |

Veri: **Digits** – 1797 el yazısı rakam, her biri 8×8 = **64 piksel** özellik.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print("Boyut:", X.shape)
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for ax, img, lbl in zip(axes, digits.images, y):
    ax.imshow(img, cmap="gray_r"); ax.set_title(lbl); ax.axis("off")
plt.show()

**Yorum:** Her görüntü 64 boyutlu bir vektör. Bu kadar boyutu doğrudan görselleştiremeyiz → boyut indirgeme gerekir.

## 1. PCA – Açıklanan varyans

In [ ]:
Xs = StandardScaler().fit_transform(X)
pca_full = PCA().fit(Xs)
kum = np.cumsum(pca_full.explained_variance_ratio_)
n95 = int(np.argmax(kum >= 0.95)) + 1

plt.figure(figsize=(8, 4))
plt.plot(range(1, 65), kum, color="tab:blue"); plt.axhline(0.95, color="r", ls="--"); plt.axvline(n95, color="r", ls="--")
plt.title(f"Kümülatif açıklanan varyans – %95 için {n95} bileşen"); plt.xlabel("Bileşen sayısı"); plt.show()

**Yorum:** 64 boyuttaki bilginin %95'i çok daha az bileşenle korunabiliyor — veri gerçekte daha düşük boyutlu bir yapıya sahip.

## 2. PCA vs t-SNE ile 2D görselleştirme

In [ ]:
X_pca = PCA(n_components=2).fit_transform(Xs)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(Xs)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, Z, t in [(ax[0], X_pca, "PCA"), (ax[1], X_tsne, "t-SNE")]:
    sc = a.scatter(*Z.T, c=y, cmap="tab10", s=8)
    a.set_title(f"{t} – 64 boyut → 2 boyut")
fig.colorbar(sc, ax=ax, ticks=range(10), label="rakam")
plt.show()

**Yorum:** PCA (doğrusal) rakamları kısmen ayırır, birçok sınıf üst üste biner. t-SNE (doğrusal olmayan) 10 rakamı neredeyse kusursuz ayrı adacıklar halinde gösterir. Ancak t-SNE yalnızca görselleştirme içindir.

## 3. PCA'yı model girdisi olarak kullanma

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
sonuc = []
for n in [2, 5, 10, 20, n95, 64]:
    pipe = Pipeline([("s", StandardScaler()), ("pca", PCA(n_components=n)),
                     ("m", LogisticRegression(max_iter=2000))]).fit(X_train, y_train)
    sonuc.append({"bileşen": n, "Test Accuracy": accuracy_score(y_test, pipe.predict(X_test))})
sonuc = pd.DataFrame(sonuc); display(sonuc.round(3))
sonuc.plot(x="bileşen", y="Test Accuracy", marker="o", figsize=(7, 3.5), color="tab:green", legend=False)
plt.title("PCA bileşen sayısı vs doğruluk"); plt.show()

**Yorum:** Az sayıda bileşenle bile tüm özelliklere yakın doğruluk elde edilir → daha hızlı ve daha az bellek kullanan model. PCA, pipeline içinde olduğu için yalnızca train verisine fit edildi.

## Özet
### Data Analysis Key Findings
- 64 piksellik varyansın %95'i çok daha az PCA bileşeniyle korunuyor.
- t-SNE görselleştirmede rakamları PCA'dan çok daha iyi ayırıyor; PCA ise model girdisi olarak az bileşenle yüksek doğruluk sağlıyor.
### Insights or Next Steps
- Büyük veride t-SNE yerine daha hızlı olan **UMAP** (`umap-learn`) deneyin.